# VietDub Auto — Colab / Kaggle

Chạy pipeline dub video (YouTube / Douyin / Bilibili) sang tiếng Việt với **GPU miễn phí**.

**Cách dùng:** Bật GPU trước khi chạy.
- **Colab:** `Runtime` → `Change runtime type` → `T4 GPU`.
- **Kaggle:** panel bên phải → `Accelerator` → `GPU T4 x2` (hoặc P100).

Sau đó chạy lần lượt từng cell từ trên xuống.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy source code

Sửa `REPO_URL` thành repo GitHub của bạn. Nếu chưa push lên GitHub, dùng cell upload zip ở mục 2b.

In [ ]:
import os

# TODO: đổi thành repo của bạn
REPO_URL = "https://github.com/<username>/<repo>.git"
PROJECT_DIR = "/content/vietdub" if os.path.exists("/content") else "/kaggle/working/vietdub"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())
!ls -la

### 2b. (Tùy chọn) Upload source dạng zip thay vì git clone

Chỉ chạy cell này nếu bạn KHÔNG dùng GitHub. Nén thư mục project thành `project.zip` rồi upload.

In [ ]:
# Colab: dùng file picker. Kaggle: thêm dataset qua 'Add Data'.
import os

if os.path.exists("/content"):
    from google.colab import files
    uploaded = files.upload()  # chọn project.zip
    !mkdir -p /content/vietdub && unzip -o project.zip -d /content/vietdub
    os.chdir("/content/vietdub")
else:
    print("Kaggle: dùng nút 'Add Data' để thêm project dưới dạng dataset, rồi copy sang /kaggle/working.")
print("Working dir:", os.getcwd())

## 3. Cài đặt phụ thuộc

Cài ffmpeg, requirements, và VieNeu-TTS. Colab có sẵn ffmpeg; Kaggle cũng có sẵn.

In [ ]:
# ffmpeg (thường đã có sẵn; cài lại cho chắc)
!apt-get -qq install -y ffmpeg > /dev/null 2>&1 || true
!ffmpeg -version | head -n 1

# Requirements của project
!pip install -q -r requirements.txt

# VieNeu-TTS (chạy GPU)
!pip install -q git+https://github.com/pnnbao97/VieNeu-TTS.git

## 4. Cấu hình API keys & tài khoản

Điền key vào đây. **Không commit key lên GitHub.**
- Bắt buộc ít nhất 1 key dịch: `OPUS_API_KEY` / `GEMINI_API_KEY` / `GROQ_API_KEY`.
- `GROQ_API_KEY` cũng dùng cho Whisper transcribe (video không có phụ đề).
- Muốn upload thì điền YouTube/Facebook; không thì để trống, video vẫn xuất ra file.

In [ ]:
import os

# --- Ngôn ngữ nguồn (video Trung) ---
os.environ["SOURCE_LANG"] = "zh"

# --- Dịch (điền ít nhất 1) ---
os.environ["OPUS_API_KEY"] = ""
os.environ["OPUS_API_URL"] = "https://api.justwoker.icu/v1/messages"
os.environ["OPUS_MODEL"] = "claude-opus-4-8"
os.environ["GEMINI_API_KEY"] = ""
os.environ["GROQ_API_KEY"] = ""  # cũng dùng cho Whisper

# --- TTS ---
os.environ["TTS_ENGINE"] = "vieneu"
os.environ["VIENEU_VOICE"] = "Hải Đăng"

# --- YouTube (tùy chọn, để trống nếu không upload) ---
os.environ["YOUTUBE_CLIENT_ID"] = ""
os.environ["YOUTUBE_CLIENT_SECRET"] = ""
os.environ["YOUTUBE_REFRESH_TOKEN"] = ""

# --- Facebook (tùy chọn) ---
os.environ["FACEBOOK_PAGE_ID"] = ""
os.environ["FACEBOOK_ACCESS_TOKEN"] = ""

print("Đã set env.")

### 4b. (Tùy chọn) Cookies cho Douyin / Bilibili / YouTube

Nếu tải bị chặn hoặc giới hạn chất lượng, export cookies từ trình duyệt (định dạng Netscape `cookies.txt`) rồi upload.

In [ ]:
import os

if os.path.exists("/content"):
    from google.colab import files
    up = files.upload()  # chọn cookies.txt
    cookie_path = os.path.join(os.getcwd(), list(up.keys())[0])
    os.environ["YT_COOKIES_FILE"] = cookie_path
    print("Cookies:", cookie_path)
else:
    # Kaggle: upload cookies.txt qua 'Add Data' rồi trỏ đường dẫn tại đây
    # os.environ["YT_COOKIES_FILE"] = "/kaggle/input/<dataset>/cookies.txt"
    print("Kaggle: thêm cookies.txt qua Add Data rồi set YT_COOKIES_FILE thủ công.")

## 5. Chạy pipeline

Dán URL video vào `VIDEO_URL`. Bật/tắt upload bằng cờ dưới.

In [ ]:
VIDEO_URL = "https://www.douyin.com/video/xxxxxxxxxx"

UPLOAD_YOUTUBE = False
UPLOAD_FACEBOOK = False

args = ["python", "backend/main.py", "--url", VIDEO_URL, "--output", "./output"]
if UPLOAD_YOUTUBE:
    args.append("--upload-youtube")
if UPLOAD_FACEBOOK:
    args.append("--upload-facebook")

import subprocess
print("Chạy:", " ".join(args))
subprocess.run(args, check=True)

## 6. Tải kết quả về máy

Video + SRT nằm trong thư mục `output/`.

In [ ]:
import os, glob

outputs = sorted(glob.glob("output/*.mp4")) + sorted(glob.glob("output/*.srt"))
print("Files:", outputs)

if os.path.exists("/content"):
    from google.colab import files
    for f in outputs:
        files.download(f)
else:
    print("Kaggle: file trong output/ đã nằm ở /kaggle/working, tải qua panel Output bên phải.")